![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG and Milvus database to work with `ibm-watsonx-ai` SDK documentation.

#### Disclaimers

- Use only Spaces that are available in the watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate the usage of IBM AutoAI RAG. The AutoAI RAG experiment conducted in this notebook uses data scraped from the `ibm-watsonx-ai` SDK documentation.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The learning goals of this notebook are:

- Create an AutoAI RAG job that will find the best RAG pattern based on provided data


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [RAG Optimizer definition](#RAG-Optimizer-definition)
3. [Run the RAG Experiment](#Run-the-RAG-Experiment)
4. [Comparison and testing of RAG Patterns](#Comparison-and-testing-of-RAG-Patterns)
5. [Historical runs](#Historical-runs)
6. [Cleanup](#Cleanup)
7. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup task:

-  Contact your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install -U "ibm-watsonx-ai[rag]>=1.4.10" | tail -n 1

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [ ]:
import os

try:
    username = os.environ["USERNAME"]
except KeyError:
    username = input("Please enter your username (hit enter): ")

try:
    url = os.environ["URL"]
except KeyError:
    url = input("Please enter the platform url (hit enter): ")

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [ ]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, you need to create a space for your work. If you do not have a space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click **New Deployment Space**
- Create an empty space
- Go to the space `Settings` tab
- Copy the `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. Find more information in the [Space Management sample notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.0/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign the space ID below

In [ ]:
try:
    space_d = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

To print all existing spaces, use the `list` method.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai, you need to set the **space** which you will be using.

In [7]:
client.set.default_space(space_id)

'SUCCESS'

<a id="RAG-Optimizer-definition"></a>
## RAG Optimizer definition

### Define a connection to the training data

Define connection information to access the COS bucket and the file that contains the training data. This example uses [`ibm_watsonx_ai`](https://ibm.github.io/watsonx-ai-python-sdk/index.html) SDK documentation content.

The following code cell downloads the `ibm_watsonx_ai` Python SDK compressed file from GitHub (if not already downloaded), and extracts its contents to a specified folder.

In [8]:
import os
import zipfile

import wget

archive_name = "watsonx-ai-python-sdk"
archive_zip = "watsonx-ai-python-sdk.zip"

if not os.path.isfile(archive_zip):
    wget.download(
        "https://github.com/IBM/watsonx-ai-python-sdk/archive/refs/heads/gh-pages.zip",
        out=archive_zip,
    )

with zipfile.ZipFile(archive_zip, "r") as zip_ref:
    zip_ref.extractall(archive_name)

Create a connection to COS.

In [ ]:
datasource_name = "bluemixcloudobjectstorage"

# Provide COS credentials
bucket_name = "PASTE YOUR BUCKET NAME HERE"
access_key = "PASTE YOUR ACCESS KEY HERE"
secret_key = "PASTE YOUR SECRET KEY HERE"
url = "PASTE YOUR URL HERE"

In [10]:
conn_meta_props = {
    client.connections.ConfigurationMetaNames.NAME: f"Connection to Database - {datasource_name} ",
    client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: client.connections.get_datasource_type_id_by_name(
        datasource_name
    ),
    client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection to external Database",
    client.connections.ConfigurationMetaNames.PROPERTIES: {
        "bucket": bucket_name,
        "access_key": access_key,
        "secret_key": secret_key,
        "iam_url": "https://iam.cloud.ibm.com/identity/token",
        "url": url,
    },
}

conn_details = client.connections.create(meta_props=conn_meta_props)
connection_id = client.connections.get_id(conn_details)

Creating connections...
SUCCESS


Create a Data Connection that represents input data references.

In [11]:
from ibm_watsonx_ai.helpers import DataConnection, S3Location

data_connection = DataConnection(
    connection_asset_id=connection_id,
    location=S3Location(bucket=bucket_name, path=archive_name),
)
input_data_references = [data_connection]
input_data_references[0].set_client(client)

Filter documents with the `.html` extension and save them to the COS bucket.

In [12]:
html_docs_files = []

for root, dirs, files in os.walk(archive_name):
    if root != f"{archive_name}/watsonx-ai-python-sdk-gh-pages":
        continue

    for file in filter(lambda x: x.endswith(".html"), files):
        file_path = os.path.join(root, file)
        html_docs_files.append(file_path)

Writing all SDK documents might take around 3 minutes.

In [13]:
for i, html_docs_file in enumerate(html_docs_files):
    data_connection.write(html_docs_file, remote_name=html_docs_file.split("/")[-1])
    print(
        f"Progress: {'✓' * (i+1)}{'.' * (len(html_docs_files)-i-1)}",
        end="\r",
        flush=True,
    )

  Using cached pyarrow-22.0.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (3.2 kB)
Using cached pyarrow-22.0.0-cp312-cp312-macosx_12_0_arm64.whl (34.2 MB)


### Define a connection to the test data

Upload a `json` file that you want to use as a benchmark to COS and then define a connection to the file. This example uses content from the [`ibm_watsonx_ai`](https://ibm.github.io/watsonx-ai-python-sdk/index.html) SDK documentation.

In [14]:
benchmarking_data_IBM_page_content = [
    {
        "question": "How to install ibm-watsonx-ai library?",
        "correct_answer": "pip install ibm-watsonx-ai",
        "correct_answer_document_ids": ["install.html"],
    },
    {
        "question": "What is Credentials class parameters?",
        "correct_answer": "url, api_key, name, iam_serviceid_crn, token, projects_token, username, password, instance_id, version, bedrock_url, proxies, verify",
        "correct_answer_document_ids": ["base.html"],
    },
    {
        "question": "How to get AutoAI pipeline with number 3?",
        "correct_answer": "get_pipeline(pipeline_name='Pipeline_3')",
        "correct_answer_document_ids": ["autoai_working_with_class_and_optimizer.html"],
    },
    {
        "question": "How to get list of Embedding Models?",
        "correct_answer": "client.foundation_models.EmbeddingModels",
        "correct_answer_document_ids": ["fm_embeddings.html"],
    },
    {
        "question": "How to retrieve the list of model lifecycle data?",
        "correct_answer": "get_model_lifecycle(url='https://us-south.ml.cloud.ibm.com', model_id='ibm/granite-13b-instruct-v2')",
        "correct_answer_document_ids": ["fm_helpers.html"],
    },
    {
        "question": "What is path to ModelInference class?",
        "correct_answer": "ibm_watsonx_ai.foundation_models.inference.ModelInference",
        "correct_answer_document_ids": ["fm_model_inference.html"],
    },
    {
        "question": "What is method for get model inference details?",
        "correct_answer": "get_details()",
        "correct_answer_document_ids": ["fm_model_inference.html"],
    },
]

Upload the benchmark testing data to the bucket as a `json` file.

In [15]:
import json

test_filename = "benchmarking_data_ibm_watson_ai.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data_IBM_page_content, json_file, indent=4)

test_asset_details = client.data_assets.create(
    name=test_filename, file_path=test_filename
)

test_asset_id = client.data_assets.get_id(test_asset_details)
test_asset_id

Creating data asset...
SUCCESS


'070e71d9-2a96-4c76-bfb8-bc93a8dcb855'

Define connection information to the testing data.

In [16]:
from ibm_watsonx_ai.helpers import DataConnection

test_data_references = [DataConnection(data_asset_id=test_asset_id)]

### Set up connectivity information to Milvus

<b>This notebook focuses on a self-managed Milvus cluster using <a href="https://cloud.ibm.com/docs/watsonxdata?topic=watsonxdata-adding-milvus-service" target="_blank" rel="noopener no referrer">IBM watsonx.data.</a></b>

The following cell retrieves the Milvus username, password, host, and port from the environment (if available) and prompts you to provide them manually in case of failure.

You can provide a connection asset ID to read all required connection data from it. Before doing so, make sure that a connection asset was created in your space.

In [ ]:
import getpass
import os

milvus_connection_id = input(
    "Provide connection asset ID in your space. Skip this, if you wish to type credentials by hand and hit enter: "
)

if not milvus_connection_id:
    try:
        username = os.environ["USERNAME"]
    except KeyError:
        username = input("Please enter your Milvus user name and hit enter: ")

    try:
        password = os.environ["PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your Milvus password and hit enter: ")

    try:
        host = os.environ["HOST"]
    except KeyError:
        host = input("Please enter your Milvus hostname and hit enter: ")

    try:
        port = os.environ["PORT"]
    except KeyError:
        port = input("Please enter your Milvus port number and hit enter: ")

    try:
        ssl = os.environ["SSL"]
    except:
        ssl = bool(
            input(
                "Please enter ('y'/anything) if your Milvus instance has SSL enabled. Skip if it is not: "
            )
        )

    # Create connection
    milvus_data_source_type_id = client.connections.get_datasource_type_uid_by_name(
        "milvus"
    )
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Milvus Connection",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: milvus_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": host,
                "port": port,
                "username": username,
                "password": password,
                "ssl": ssl,
            },
        }
    )

    milvus_connection_id = client.connections.get_id(details)

Creating connections...
SUCCESS


Define connection information to vector store references.

In [18]:
vector_store_references = [DataConnection(connection_asset_id=milvus_connection_id)]

### RAG Optimizer configuration

Provide the input information for the AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [19]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.foundation_models.schema import AutoAIRAGRetrievalConfig

experiment = AutoAI(
    credentials=credentials,
    space_id=space_id,
)

retrieval_config = AutoAIRAGRetrievalConfig(
    method="window",
    number_of_chunks=6,
    window_size=2,
)

chunking_config = {"method": "recursive", "chunk_size": 1024, "chunk_overlap": 256}

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG test - sample noteook",
    description="Experiment run in sample notebook",
    chunking=[chunking_config],
    retrieval=[retrieval_config],
    max_number_of_rag_patterns=6,
    optimization_metrics=[AutoAI.RAGMetrics.ANSWER_CORRECTNESS],
)

To retrieve the configuration parameters, use `get_params()`.

In [20]:
rag_optimizer.get_params()

{'name': 'AutoAI RAG test - sample noteook',
 'description': 'Experiment run in sample notebook',
 'chunking': [{'method': 'recursive',
   'chunk_size': 1024,
   'chunk_overlap': 256}],
 'max_number_of_rag_patterns': 6,
 'optimization_metrics': ['answer_correctness'],
 'retrieval': [{'method': 'window', 'number_of_chunks': 6, 'window_size': 2}]}

<a id="Run-the-RAG-Experiment"></a>
## Run the RAG Experiment

Call the `run()` method to trigger the AutoAI RAG experiment. Choose one of two modes: 

- To use the **interactive mode** (synchronous job), specify `background_mode=False` 
- To use the **background mode** (asynchronous job), specify `background_mode=True`

In [21]:
run_details = rag_optimizer.run(
    input_data_references=input_data_references,
    test_data_references=test_data_references,
    vector_store_references=vector_store_references,
    background_mode=False,
)



##############################################

Running 'b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca'

##############################################


pending.............
running...........................................................
completed
Training of 'b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca' finished successfully.


To monitor the AutoAI RAG jobs in background mode, use the `get_run_status()` method.

In [22]:
rag_optimizer.get_run_status()

'completed'

<a id="Comparison-and-testing-of-RAG-Patterns"></a>
## Comparison and testing of RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. Use the DataFrame to compare all discovered patterns and select the one you want for further testing.

In [23]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern3,0.6879,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-8b-instruct-intel,sequential
Pattern1,0.5666,recursive,1024,256,ibm/slate-125m-english-rtrvr-v2,cosine,window,6,ibm/granite-3-8b-instruct,sequential
Pattern2,0.5666,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-3-8b-instruct-intel,sequential
Pattern4,0.5666,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-8b-instruct,sequential
Pattern5,0.5666,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-8b-instruct-intel,sequential
Pattern6,0.5666,recursive,1024,256,ibm/slate-125m-english-rtrvr-v2,cosine,window,6,ibm/granite-3-8b-instruct-intel,sequential


Additionally, you can pass the `scoring` parameter to the summary method to filter RAG patterns, starting with the best.

In [24]:
summary = rag_optimizer.summary(scoring="faithfulness")

### Get the selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [25]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern()

Best pattern is: Pattern3


To retrieve the pattern details, use the `get_pattern_details` method.

In [26]:
rag_optimizer.get_pattern_details()

{'composition_steps': ['model_selection',
  'chunking',
  'embeddings',
  'retrieval',
  'generation'],
 'duration_seconds': 8,
 'location': {'evaluation_results': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/evaluation_results.json',
  'indexing_notebook': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/indexing_notebook.ipynb',
  'indexing_service_code': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/indexing_ai_service.gz',
  'indexing_service_metadata': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/indexing_service_metadata.json',
  'inference_notebook': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/inference_notebook.ipynb',
  'inference_service_code': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/inference_ai_service.gz',
  'inference_service_metadata': 'default_autoai_rag_out/b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca/Pattern3/inference_service_

Query the RAGPattern locally to test it.

In [27]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)

context = RuntimeContext(
    api_client=client,
    request_payload_json={
        "messages": [
            {
                "role": "user",
                "content": "How to use new approach of providing credentials to APIClient?",
            }
        ]
    },
)

resp = best_pattern.inference_service(runtime_context)[0](context)

In [28]:
print(resp["body"]["choices"][0]["message"]["content"])

To use the new approach of providing credentials to APIClient, you would typically follow these steps:

1. First, ensure that you have the necessary permissions and access rights to the API you're trying to connect to.

2. Next, you'll need to obtain your API credentials. These usually come in the form of an API key or token. The process for obtaining these can vary depending on the specific API service you're using.

3. Once you have your credentials, you'll need to set up your APIClient. This usually involves initializing a new instance of the APIClient class and passing your credentials to it. The exact method of doing this can depend on the specific implementation of APIClient you're using.

4. After setting up your APIClient with your credentials, you can then use it to interact with the API. This might involve calling specific methods on the APIClient instance to send requests and receive responses from the API.


### Deploy the RAGPattern

To deploy the RAGPattern, store the defined RAG function and then create a deployed asset.

In [29]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentataion", space_id=space_id
)



######################################################################################

Synchronous deployment creation for id: 'a4301e61-f8d2-4900-a53b-3ce1aa58f555' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
.....
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='19f44eab-68f8-4162-baa6-c0c4e08fe98e'
-----------------------------------------------------------------------------------------------




### Test the deployed function

The RAG service is now deployed in our space. To test the solution, run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [30]:
deployment_id = client.deployments.get_id(deployment_details)

question = "How to add Task Credentials?"

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)

In [31]:
print(score_response["choices"][0]["message"]["content"])

To add task credentials, you would typically follow these steps:

1. Navigate to the relevant section of your system. This might be under "Data Connections" or "Task Settings", depending on the specific platform you're using.

2. Look for an option related to "Credentials" or "Authentication". This is usually where you can manage and add new credentials.

3. Click on "Add" or "New Credential" to create a new set of credentials.

4. Fill in the necessary details. This typically includes:
   - Name: A descriptive name for the credential.
   - Type: The type of credential, such as "API Key", "Username/Password", etc.
   - Value: The actual credential value. For an API key, this would be your unique key. For a username/password, this would be your login credentials.

5. Save the new credential.

Please note that the exact steps may vary depending on the specific system or platform you're using. If you provide more details about your system, I can give a more precise guide.

For a more deta

<a id="Historical-runs"></a>
## Historical runs

In this section, you will learn how to work with historical RAG Optimizer jobs (runs).

To list historical runs, use the `list()` method and provide the `'rag_optimizer'` filter.

In [ ]:
experiment.runs(filter="rag_optimizer").list()

In [33]:
run_id = run_details["metadata"]["id"]
run_id

'b6a1ce14-cad3-4f30-8afd-bc2ea6d155ca'

### Get the executed optimizer's configuration parameters

In [34]:
experiment.runs.get_rag_params(run_id=run_id)

{'name': 'AutoAI RAG test - sample noteook',
 'description': 'Experiment run in sample notebook',
 'chunking': [{'chunk_overlap': 256,
   'chunk_size': 1024,
   'method': 'recursive'}],
 'max_number_of_rag_patterns': 6,
 'retrieval': [{'method': 'window', 'number_of_chunks': 6, 'window_size': 2}],
 'optimization_metrics': ['answer_correctness']}

### Get the historical rag_optimizer instance and training details

In [35]:
historical_opt = experiment.runs.get_rag_optimizer(run_id)

### List trained patterns for the selected optimizer

In [36]:
historical_opt.summary()

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern3,0.6879,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-8b-instruct-intel,sequential
Pattern1,0.5666,recursive,1024,256,ibm/slate-125m-english-rtrvr-v2,cosine,window,6,ibm/granite-3-8b-instruct,sequential
Pattern2,0.5666,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-3-8b-instruct-intel,sequential
Pattern4,0.5666,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-8b-instruct,sequential
Pattern5,0.5666,recursive,1024,256,ibm/granite-embedding-278m-multilingual,cosine,window,6,ibm/granite-3-8b-instruct-intel,sequential
Pattern6,0.5666,recursive,1024,256,ibm/slate-125m-english-rtrvr-v2,cosine,window,6,ibm/granite-3-8b-instruct-intel,sequential


<a id="Cleanup"></a>
## Cleanup

To delete the current experiment, use the `cancel_run(hard_delete=True)` method.

**Warning:** Be careful: once you delete an experiment, you will no longer be able to refer to it.

In [37]:
rag_optimizer.cancel_run(hard_delete=True)

'SUCCESS'

To delete the deployment, use the `delete` method. 

**Warning:** If you keep the deployment active, it might lead to unnecessary consumption of Compute Unit Hours (CUHs).

In [38]:
client.deployments.delete(deployment_id)

'SUCCESS'

To clean up all of the created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

follow the steps in this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.1/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI RAG experiments. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Paweł Kocur**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.